In [1]:
import sys
sys.path.append("../src")

import numpy as np

from bb84 import run_bb84_with_eve
from error_correction import parity_error_correction
from privacy_amplification import derive_final_key_bits, key_fingerprint
from encryption import run_secure_message_exchange, message_to_bits

In [2]:
bb84_result = run_bb84_with_eve(
    n=10000,
    eve_intercept_prob=0.05
)

raw_alice_key = bb84_result["alice_key"]
raw_bob_key = bb84_result["bob_key"]

print("QBER:", bb84_result["qber"])
print("Raw key length:", len(raw_alice_key))
print("Raw mismatches:", np.sum(raw_alice_key != raw_bob_key))

QBER: 0.01402439024390244
Raw key length: 4920
Raw mismatches: 69


In [3]:
correction_result = parity_error_correction(
    raw_alice_key,
    raw_bob_key,
    block_size=16,
    passes=5
)

alice_reconciled = correction_result["corrected_alice_key"]
bob_reconciled = correction_result["corrected_bob_key"]

print("Reconciled key length:", len(alice_reconciled))
print("Final mismatches:", correction_result["final_mismatches"])
print("Corrections applied:", correction_result["corrections_applied"])

Reconciled key length: 4920
Final mismatches: 0
Corrections applied: 69


In [4]:
message = "Hello quantum world"
message_bits = message_to_bits(message)

final_alice_key = derive_final_key_bits(
    alice_reconciled,
    output_length_bits=len(message_bits)
)

final_bob_key = derive_final_key_bits(
    bob_reconciled,
    output_length_bits=len(message_bits)
)

print("Message bits required:", len(message_bits))
print("Final Alice key length:", len(final_alice_key))
print("Final Bob key length:", len(final_bob_key))
print("Final keys match:", np.array_equal(final_alice_key, final_bob_key))
print("Final key fingerprint:", key_fingerprint(final_alice_key))

Message bits required: 152
Final Alice key length: 152
Final Bob key length: 152
Final keys match: True
Final key fingerprint: 1cd0622f32db2fe8


In [5]:
result = run_secure_message_exchange(
    message="Hello quantum world",
    n_qubits=10000,
    eve_intercept_prob=0.05,
    qber_threshold=0.11,
    use_error_correction=True,
    error_correction_block_size=16,
    error_correction_passes=5,
    use_privacy_amplification=True,
    privacy_compression_ratio=0.5
)

print("Status:", result["status"])
print("Reason:", result["reason"])
print("QBER:", result["qber"])
print("Raw mismatches:", result["raw_mismatches"])
print("Final mismatches:", result["final_mismatches"])
print("Privacy amplification used:", result["privacy_amplification_used"])
print("Final key fingerprint:", result["final_key_fingerprint"])
print("Original message:", result["message"])
print("Decrypted message:", result["decrypted_message"])

Status: success
Reason: Message encrypted and decrypted successfully.
QBER: 0.01178318931657502
Raw mismatches: 60
Final mismatches: 0
Privacy amplification used: True
Final key fingerprint: 26bca0676fbee67b
Original message: Hello quantum world
Decrypted message: Hello quantum world


## Privacy Amplification

This notebook adds a final key derivation step after BB84 key generation and parity-based correction.

After correction, Alice and Bob share matching reconciled keys. Privacy amplification compresses this reconciled key into a shorter final key. This reduces the usefulness of any partial information Eve may have gained during transmission or public reconciliation.

The project uses SHAKE-256 to derive a final key of the required message length, while requiring enough reconciled key material according to the selected compression ratio.